# Блок 5. Практика: Python для обработки данных

В этом notebook:
- Занятие 5.1: Polars (DataFrame, LazyFrame, выражения)
- Занятие 5.2: Энтропия и работа с IP-адресами
- Занятие 5.3: Машинное обучение для обнаружения аномалий

In [ ]:
import polars as pl
import math
from collections import Counter
import ipaddress

# Путь к данным из предыдущих блоков
PARQUET_PATH = "../lesson03/data/auth_events.parquet"

## Занятие 5.1: Polars

### DataFrame vs LazyFrame

In [ ]:
# DataFrame — загружает все данные в память
df = pl.read_parquet(PARQUET_PATH)
print(f"DataFrame: {df.shape[0]:,} строк, {df.shape[1]} колонок")
print(df.head())

In [ ]:
# LazyFrame — отложенные вычисления (оптимизируется перед выполнением)
lf = pl.scan_parquet(PARQUET_PATH)

# Строим цепочку операций
result = (
    lf
    .filter(pl.col("success") == False)
    .group_by("source_ip")
    .agg(pl.count().alias("failed_count"))
    .sort("failed_count", descending=True)
    .limit(10)
    .collect()  # Выполняется только здесь
)
print(result)

### Выражения и трансформации

In [ ]:
# Добавляем временные колонки
df_enriched = df.with_columns(
    pl.col("timestamp").dt.hour().alias("hour"),
    pl.col("timestamp").dt.weekday().alias("weekday"),
    pl.col("timestamp").dt.date().alias("date"),
    # Флаг нерабочего времени (до 9 или после 18)
    ((pl.col("timestamp").dt.hour() < 9) | (pl.col("timestamp").dt.hour() > 18))
    .alias("is_non_working_hours")
)
print(df_enriched.head())

## Занятие 5.2: Энтропия и работа с IP

### Энтропия Шеннона для DGA-детекции

In [ ]:
def shannon_entropy(s: str) -> float:
    """Вычисляет энтропию Шеннона для строки."""
    if not s:
        return 0.0
    freq = Counter(s)
    length = len(s)
    return -sum((count / length) * math.log2(count / length) for count in freq.values())

# Тестируем на примерах
print("Энтропия примеров:")
print(f"  'google':     {shannon_entropy('google'):.3f} (низкая)")
print(f"  'x7kj2m9p':   {shannon_entropy('x7kj2m9p'):.3f} (высокая - DGA)")
print(f"  'aaaaaa':     {shannon_entropy('aaaaaa'):.3f} (минимальная)")

### Работа с IP-адресами

In [ ]:
def is_internal_ip(ip_str: str) -> bool:
    """Проверяет, является ли IP внутренним (RFC1918)."""
    try:
        ip = ipaddress.ip_address(ip_str)
        return ip.is_private
    except ValueError:
        return False

# Добавляем флаг внутреннего IP к данным
df_with_ip_info = df.with_columns(
    pl.col("source_ip")
    .cast(pl.String)
    .map_elements(is_internal_ip, return_dtype=pl.Boolean)
    .alias("is_internal_ip")
)

# Статистика по типу IP
print(df_with_ip_info.group_by("is_internal_ip").agg(pl.count().alias("count")))

## Занятие 5.3: Машинное обучение

### Feature Engineering

In [ ]:
# Извлекаем признаки для каждого пользователя
user_features = df.group_by("username").agg(
    pl.count().alias("total_events"),
    pl.col("success").mean().alias("success_rate"),
    pl.col("source_ip").n_unique().alias("unique_ips"),
    ((pl.col("timestamp").dt.hour() < 9) | (pl.col("timestamp").dt.hour() > 18))
    .mean().alias("non_working_ratio")
)
print(user_features.head(10))

### Isolation Forest для обнаружения аномалий

In [ ]:
from sklearn.ensemble import IsolationForest
import numpy as np

# Подготовка признаков (заполняем NaN нулями)
feature_cols = ["total_events", "success_rate", "unique_ips", "non_working_ratio"]
X = user_features.select(feature_cols).fill_null(0).to_numpy()

# Обучение модели
model = IsolationForest(n_estimators=100, contamination=0.1, random_state=42)
predictions = model.fit_predict(X)

# Добавляем результат к данным
user_features_with_anomaly = user_features.with_columns(
    pl.Series("is_anomaly", predictions == -1)
)

print(f"Найдено аномалий: {(predictions == -1).sum()} из {len(predictions)}")
print("\nАномальные пользователи:")
print(user_features_with_anomaly.filter(pl.col("is_anomaly")))

## Итоги

В этом блоке мы освоили:

1. **Polars**: быстрая альтернатива pandas с LazyFrame и выражениями
2. **Энтропия**: математический инструмент для DGA-детекции
3. **IP-адреса**: классификация внутренних и внешних адресов
4. **Isolation Forest**: обнаружение аномалий без размеченных данных

**Ключевые применения в ИБ:**
- ETL-пайплайны для обработки логов
- Детекция DGA-доменов по энтропии
- Поиск аномальных пользователей и хостов